In [1]:
#Apply otliers removal
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.cluster import Birch
from sklearn.cluster import OPTICS
#from scipy.stats import zscore
import numpy as np
import time
# Load dataset
df= pd.read_csv("data/data.csv")

In [2]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['diagnosis'] = le.fit_transform(df['diagnosis'])
df.isnull().sum().sum()
df = df.dropna(axis=1, how='all')


In [3]:

# Drop target and ID columns
X = df.drop(columns=["id"], errors="ignore")
print("Features shape:", X.shape)

Features shape: (569, 31)


In [4]:
# Compute z-scores
z_scores = np.abs((X - X.mean()) / X.std())

# Define threshold
threshold = 3
# Get row indices where ANY feature is an outlier
outlier_indices = np.where((z_scores > threshold).any(axis=1))[0]

# Remove them
X_clean = X.drop(index=X.index[outlier_indices])
print("Original features shape:", X.shape)
print("After Outlier Removal:", X_clean.shape)


Original features shape: (569, 31)
After Outlier Removal: (495, 31)


In [6]:
#Apply StandardScaler
scaler = StandardScaler()
X_so = scaler.fit_transform(X_clean)
print("Scaled shape:", X_so.shape)

Scaled shape: (495, 31)


In [7]:
#Define Cluster Parameters
k_values = range(2, 9)  # clusters 2–8 for KMeans, GMM, Agglomerative, Spectral
n_init = 10  # random initialization for KMeans, GMM, Spectral
dbscan_eps = [0.5, 1.0, 1.5]  # DBSCAN eps values
min_samples = 5

#Function to Compute Metrics
def compute_metrics(X_data, labels):
    sil = silhouette_score(X_data, labels)
    db = davies_bouldin_score(X_data, labels)
    ch = calinski_harabasz_score(X_data, labels)
    return sil, db, ch

In [8]:
#K-Means on Scaled + OutlierRemoval Data
start_time = time.time()
kmean_outliers = []
for k in k_values:
    km = KMeans(n_clusters=k, n_init=n_init, random_state=42)
    labels = km.fit_predict(X_so)
    sil, db, ch = compute_metrics(X_so, labels)
    kmean_outliers.append({"algorithm": "KMeans", "preprocessing": "OutlierRemoval", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
end_time = time.time()
runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"KMeans runtime: {runtime:.4f} seconds")   


Runtime: 4.631794214248657 seconds
KMeans runtime: 4.6318 seconds


In [9]:
#Gaussian Mixture (GMM)on Scaled + OutlierRemoval Data
start_time = time.time()
gmm_outliers = []
for k in k_values:
    gmm = GaussianMixture(n_components=k, n_init=n_init, random_state=42)
    labels = gmm.fit_predict(X_so)
    sil, db, ch = compute_metrics(X_so, labels)
    gmm_outliers.append({"algorithm": "GMM", "preprocessing": "OutlierRemoval", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
end_time = time.time()
runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"GMM runtime: {runtime:.4f} seconds")   

Runtime: 3.7622358798980713 seconds
GMM runtime: 3.7622 seconds


In [10]:
#Agglomerative Clustering on Scaled + OutlierRemoval Data
start_time = time.time()
agg_outliers = []
for k in k_values:
    agg = AgglomerativeClustering(n_clusters=k, linkage="ward")
    labels = agg.fit_predict(X_so)
    sil, db, ch = compute_metrics(X_so, labels)
    agg_outliers.append({"algorithm": "Agglomerative", "preprocessing": "OutlierRemoval", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
end_time = time.time()
runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"Agglomerative runtime: {runtime:.4f} seconds")   

Runtime: 0.24160480499267578 seconds
Agglomerative runtime: 0.2416 seconds


In [11]:
#Spectral Clustering on Scaled + OutlierRemoval Data
start_time = time.time()
spec_outliers = []
for k in k_values:
    spec = SpectralClustering(n_clusters=k, affinity="nearest_neighbors")
    labels = spec.fit_predict(X_so)
    sil, db, ch = compute_metrics(X_so, labels)
    spec_outliers.append({"algorithm": "Spectral", "preprocessing": "OutlierRemoval", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
end_time = time.time()
runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"Spectral runtime: {runtime:.4f} seconds")   

Runtime: 0.5551624298095703 seconds
Spectral runtime: 0.5552 seconds


In [12]:
#DBSCAN on Scaled + OutlierRemoval Data
start_time = time.time()
dbscan_outliers = []
for eps in dbscan_eps:
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    labels = dbscan.fit_predict(X_so)
    
    # Remove noise points (-1)
    mask = labels != -1
    if np.sum(mask) > 1 and len(set(labels[mask])) > 1:
        sil, db, ch = compute_metrics(X_so[mask], labels[mask])
        dbscan_outliers.append({"algorithm": "DBSCAN", "preprocessing": "OutlierRemoval", "eps": eps, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
end_time = time.time()
runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"DBSCAN runtime: {runtime:.4f} seconds")

Runtime: 0.025752544403076172 seconds
DBSCAN runtime: 0.0258 seconds


In [19]:
#BIRCH on Scaled + OutlierRemoval Data
start_time = time.time()
birch_outliers = []
threshold_values = [0.2, 0.5, 1.0, 1.5]

for t in threshold_values:
    birch = Birch(n_clusters=None, threshold=t)
    labels = birch.fit_predict(X_so)

    n_clusters = len(set(labels))
    if 1 < n_clusters < len(X_so) and len(set(labels[mask])) > 1:
        sil, db, ch = compute_metrics(X_so, labels)
        birch_outliers.append({
            "algorithm": "BIRCH",
            "preprocessing": "OutliersRemoval",
            "threshold": t,
            "n_clusters": len(set(labels)),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })

end_time = time.time()
runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"BIRCH runtime: {runtime:.4f} seconds")

Runtime: 0.09327507019042969 seconds
BIRCH runtime: 0.0933 seconds


In [15]:
#OPTICS on Scaled + OutlierRemoval Data
start_time = time.time()
optics_outliers = []
min_samples_values = [3, 5, 10, 20]

for m in min_samples_values:
    optics = OPTICS(min_samples=m, xi=0.05, n_jobs=-1)
    labels = optics.fit_predict(X_so)

    # Remove noise points (-1) if needed
    unique_labels = set(labels) - {-1}

    if len(unique_labels) > 1:
        sil, db, ch = compute_metrics(X_so, labels)
        optics_outliers.append({
            "algorithm": "OPTICS",
            "preprocessing": "OutliersRemoval",
            "min_samples": m,
            "xi": 0.05,
            "n_clusters": len(unique_labels),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })

end_time = time.time()
runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"Optics runtime: {runtime:.4f} seconds")

Runtime: 5.1354687213897705 seconds
Optics runtime: 5.1355 seconds


In [ ]:
import csv


breast_cancer_results_outliers = (kmean_outliers + gmm_outliers + agg_outliers + spec_outliers + dbscan_outliers+birch_outliers + optics_outliers)

keys = ["algorithm", "preprocessing","k", "eps", "min_samples", "threshold","n_clusters","silhouette", "davies_bouldin", "calinski_harabasz"]

with open('updated_data/breast_cancer_data/breast_cancer_outliers.csv', 'w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=keys, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(breast_cancer_results_outliers)